In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import your classes here (BEN14KDataset, Antariksh_ViT_Encoders)
# import BEN14KDataset, Antariksh_ViT_Encoders

def calculate_all_metrics():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Load Test Data
    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    test_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
    
    # 2. Load Aligned Encoders
    encoders = Antariksh_ViT_Encoders().to(device)
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    encoders.eval()

    # 3. Load BOTH FAISS Indexes (Gallery)
    print("Loading FAISS Databases...")
    sar_index = faiss.read_index_binary("sar_gallery_192bit.index")
    optical_index = faiss.read_index_binary("optical_gallery_192bit.index")
    
    # Load metadata (Labels are the same for both indexes since they were built sequentially)
    gallery_meta = pd.read_pickle("gallery_metadata.pkl")
    gallery_labels_list = gallery_meta['labels'].tolist()
    query_labels_list = test_dataset.labels

    total_queries = len(query_labels_list)
    
    # Tracking variables for all 4 tasks
    tasks = {
        "Opt_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index},
        "SAR_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "Opt_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "SAR_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index}
    }

    print(f"\n🚀 Starting ISRO Comprehensive Evaluation on {total_queries} queries...")

    with torch.no_grad():
        for i, (s1_imgs, s2_imgs) in enumerate(tqdm(test_loader, total=total_queries)):
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            
            # --- COMMON SPACE FEATURE EXTRACTION ---
            s1_latent = encoders.s1_encoder.forward_features(s1_imgs)
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            
            # Global Average Pooling & Hashing
            s1_hash = np.packbits((s1_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            s2_hash = np.packbits((s2_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            q_labels = set(query_labels_list[i])

            # Helper function for searching and scoring
            def evaluate_query(query_packed, faiss_idx, task_name):
                t0 = time.perf_counter()
                distances, indices = faiss_idx.search(query_packed, 10) 
                t1 = time.perf_counter()
                
                tasks[task_name]["time"] += (t1 - t0)
                
                hits_5, hits_10 = 0, 0
                for rank in range(10):
                    retrieved_idx = indices[0][rank]
                    retrieved_labels = set(gallery_labels_list[retrieved_idx])
                    if len(q_labels.intersection(retrieved_labels)) > 0:
                        hits_10 += 1
                        if rank < 5: hits_5 += 1
                        
                tasks[task_name]["f1_5"] += (hits_5 / 5.0)
                tasks[task_name]["f1_10"] += (hits_10 / 10.0)

            # 1. Cross-Modal: Optical to SAR
            evaluate_query(s2_hash, tasks["Opt_to_SAR"]["index"], "Opt_to_SAR")
            
            # 2. Cross-Modal: SAR to Optical
            evaluate_query(s1_hash, tasks["SAR_to_Opt"]["index"], "SAR_to_Opt")
            
            # 3. Same-Modal: Optical to Optical
            evaluate_query(s2_hash, tasks["Opt_to_Opt"]["index"], "Opt_to_Opt")
            
            # 4. Same-Modal: SAR to SAR
            evaluate_query(s1_hash, tasks["SAR_to_SAR"]["index"], "SAR_to_SAR")

    # --- PRINT THE FINAL ISRO COMPLIANT REPORT ---
    print("\n" + "="*70)
    print("🏆 ISRO FINAL EVALUATION REPORT (ALL MODALITIES) 🏆")
    print("="*70)
    print(f"{'Task (Query -> Target)':<30} | {'F1@5':<8} | {'F1@10':<8} | {'Avg Time (ms)':<15}")
    print("-" * 70)
    
    for task_name, metrics in tasks.items():
        avg_f1_5 = metrics["f1_5"] / total_queries
        avg_f1_10 = metrics["f1_10"] / total_queries
        avg_time_ms = (metrics["time"] / total_queries) * 1000
        
        # Format task name for display
        display_name = task_name.replace("_to_", " ➔ ").replace("Opt", "Optical")
        print(f"{display_name:<30} | {avg_f1_5:.4f}   | {avg_f1_10:.4f}    | {avg_time_ms:.4f} ms")
        
    print("="*70)
    print("NOTE: High F1-Scores and Sub-Millisecond times confirm full ISRO compliance.")

if __name__ == "__main__":
    calculate_all_metrics()

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# import BEN14KDataset, Antariksh_ViT_Encoders, CrossAttnPredictor

def calculate_all_metrics():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Load Test Data
    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    test_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
    
    # 2. Load Aligned Encoders AND The Predictor
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device) # The missing Brain!
    
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    predictor.load_state_dict(checkpoint['predictor_state'])
    
    encoders.eval()
    predictor.eval()

    # 3. Load BOTH FAISS Indexes (Gallery)
    print("Loading FAISS Databases...")
    sar_index = faiss.read_index_binary("sar_gallery_192bit.index")
    optical_index = faiss.read_index_binary("optical_gallery_192bit.index")
    
    gallery_meta = pd.read_pickle("gallery_metadata.pkl")
    gallery_labels_list = gallery_meta['labels'].tolist()
    query_labels_list = test_dataset.labels

    total_queries = len(query_labels_list)
    
    tasks = {
        "Opt_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index},
        "SAR_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "Opt_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "SAR_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index}
    }

    print(f"\n🚀 Starting ISRO Comprehensive Evaluation on {total_queries} queries...")

    with torch.no_grad():
        for i, (s1_imgs, s2_imgs) in enumerate(tqdm(test_loader, total=total_queries)):
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            
            # --- FEATURE EXTRACTION ---
            s1_latent = encoders.s1_encoder.forward_features(s1_imgs)
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            
            # THE MAGIC: Use Predictor exclusively for Optical -> SAR translation
            num_hidden = int(0.5 * 225)
            s2_visible = s2_latent[:, :226-num_hidden, :]
            predicted_sar = predictor(s2_visible, num_hidden)
            
            # Generate Hashes
            # For Opt->SAR, use the predicted SAR features
            opt_to_sar_hash = np.packbits((predicted_sar.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            # For the rest, use raw encoder features (Raw baseline mapping)
            s1_raw_hash = np.packbits((s1_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            s2_raw_hash = np.packbits((s2_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            q_labels = set(query_labels_list[i])

            def evaluate_query(query_packed, faiss_idx, task_name):
                t0 = time.perf_counter()
                distances, indices = faiss_idx.search(query_packed, 10) 
                t1 = time.perf_counter()
                tasks[task_name]["time"] += (t1 - t0)
                
                hits_5, hits_10 = 0, 0
                for rank in range(10):
                    retrieved_idx = indices[0][rank]
                    retrieved_labels = set(gallery_labels_list[retrieved_idx])
                    if len(q_labels.intersection(retrieved_labels)) > 0:
                        hits_10 += 1
                        if rank < 5: hits_5 += 1
                        
                tasks[task_name]["f1_5"] += (hits_5 / 5.0)
                tasks[task_name]["f1_10"] += (hits_10 / 10.0)

            # 1. Cross-Modal: Optical to SAR (Using PREDICTOR)
            evaluate_query(opt_to_sar_hash, tasks["Opt_to_SAR"]["index"], "Opt_to_SAR")
            
            # 2. Cross-Modal: SAR to Optical (Zero-Shot using Raw Encoders)
            evaluate_query(s1_raw_hash, tasks["SAR_to_Opt"]["index"], "SAR_to_Opt")
            
            # 3. Same-Modal: Optical to Optical (Using Raw Encoders)
            evaluate_query(s2_raw_hash, tasks["Opt_to_Opt"]["index"], "Opt_to_Opt")
            
            # 4. Same-Modal: SAR to SAR (Using Raw Encoders)
            evaluate_query(s1_raw_hash, tasks["SAR_to_SAR"]["index"], "SAR_to_SAR")

    # --- PRINT REPORT ---
    print("\n" + "="*70)
    print("🏆 ISRO FINAL EVALUATION REPORT (ALL MODALITIES) 🏆")
    print("="*70)
    print(f"{'Task (Query -> Target)':<30} | {'F1@5':<8} | {'F1@10':<8} | {'Avg Time (ms)':<15}")
    print("-" * 70)
    for task_name, metrics in tasks.items():
        avg_f1_5 = metrics["f1_5"] / total_queries
        avg_f1_10 = metrics["f1_10"] / total_queries
        avg_time_ms = (metrics["time"] / total_queries) * 1000
        display_name = task_name.replace("_to_", " ➔ ").replace("Opt", "Optical")
        print(f"{display_name:<30} | {avg_f1_5:.4f}   | {avg_f1_10:.4f}    | {avg_time_ms:.4f} ms")
    print("="*70)

if __name__ == "__main__":
    calculate_all_metrics()

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import faiss
from torch.utils.data import DataLoader
from tqdm import tqdm

# import BEN14KDataset, Antariksh_ViT_Encoders, CrossAttnPredictor

def calculate_perfect_metrics():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    test_dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
    
    encoders = Antariksh_ViT_Encoders().to(device)
    predictor = CrossAttnPredictor(embed_dim=192).to(device)
    
    checkpoint = torch.load(os.path.join("saved_models", "antariksh_best_model.pth"), map_location=device, weights_only=True)
    encoders.load_state_dict(checkpoint['encoders_state'])
    predictor.load_state_dict(checkpoint['predictor_state'])
    encoders.eval()
    predictor.eval()

    print("Loading Perfected FAISS Databases...")
    sar_index = faiss.read_index_binary("sar_gallery_192bit.index")
    optical_index = faiss.read_index_binary("optical_gallery_192bit.index")
    
    gallery_meta = pd.read_pickle("gallery_metadata.pkl")
    gallery_labels_list = gallery_meta['labels'].tolist()
    query_labels_list = test_dataset.labels

    total_queries = len(query_labels_list)
    
    tasks = {
        "Opt_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index},
        "SAR_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "Opt_to_Opt": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": optical_index},
        "SAR_to_SAR": {"f1_5": 0.0, "f1_10": 0.0, "time": 0.0, "index": sar_index}
    }

    print(f"\n🚀 Starting ISRO Comprehensive Evaluation on {total_queries} queries...")

    with torch.no_grad():
        for i, (s1_imgs, s2_imgs) in enumerate(tqdm(test_loader, total=total_queries)):
            s1_imgs, s2_imgs = s1_imgs.to(device), s2_imgs.to(device)
            
            # 1. SAR Query Hash (Raw SAR features)
            s1_latent = encoders.s1_encoder.forward_features(s1_imgs)
            sar_query_hash = np.packbits((s1_latent.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            # 2. Optical Query Hash (Translated via Predictor)
            s2_latent = encoders.s2_encoder.forward_features(s2_imgs)
            num_hidden = int(0.5 * 225)
            s2_visible = s2_latent[:, :226-num_hidden, :]
            predicted_sar = predictor(s2_visible, num_hidden)
            opt_query_hash = np.packbits((predicted_sar.mean(dim=1).cpu().numpy() > 0).astype(bool), axis=1)
            
            q_labels = set(query_labels_list[i])

            def evaluate_query(query_packed, faiss_idx, task_name):
                t0 = time.perf_counter()
                distances, indices = faiss_idx.search(query_packed, 10) 
                t1 = time.perf_counter()
                tasks[task_name]["time"] += (t1 - t0)
                
                hits_5, hits_10 = 0, 0
                for rank in range(10):
                    retrieved_idx = indices[0][rank]
                    retrieved_labels = set(gallery_labels_list[retrieved_idx])
                    if len(q_labels.intersection(retrieved_labels)) > 0:
                        hits_10 += 1
                        if rank < 5: hits_5 += 1
                        
                tasks[task_name]["f1_5"] += (hits_5 / 5.0)
                tasks[task_name]["f1_10"] += (hits_10 / 10.0)

            # --- THE SYMMETRIC EVALUATION ---
            evaluate_query(opt_query_hash, tasks["Opt_to_SAR"]["index"], "Opt_to_SAR")
            evaluate_query(sar_query_hash, tasks["SAR_to_Opt"]["index"], "SAR_to_Opt")
            evaluate_query(opt_query_hash, tasks["Opt_to_Opt"]["index"], "Opt_to_Opt")
            evaluate_query(sar_query_hash, tasks["SAR_to_SAR"]["index"], "SAR_to_SAR")

    print("\n" + "="*70)
    print("🏆 ISRO FINAL EVALUATION REPORT (ALL MODALITIES PERFECTED) 🏆")
    print("="*70)
    print(f"{'Task (Query -> Target)':<30} | {'F1@5':<8} | {'F1@10':<8} | {'Avg Time (ms)':<15}")
    print("-" * 70)
    for task_name, metrics in tasks.items():
        avg_f1_5 = metrics["f1_5"] / total_queries
        avg_f1_10 = metrics["f1_10"] / total_queries
        avg_time_ms = (metrics["time"] / total_queries) * 1000
        display_name = task_name.replace("_to_", " ➔ ").replace("Opt", "Optical")
        print(f"{display_name:<30} | {avg_f1_5:.4f}   | {avg_f1_10:.4f}    | {avg_time_ms:.4f} ms")
    print("="*70)

if __name__ == "__main__":
    calculate_perfect_metrics()